# Automated Answer Evaluation — Analysis
Visualizes similarity scores, predicted vs actual scores, and error distribution.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import json
import csv
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [ ]:
# Load results
with open('../results/outputs.json') as f:
    outputs = json.load(f)

metrics = []
with open('../results/evaluation_metrics.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        metrics.append(row)

print(f'Loaded {len(outputs)} test results, {len(metrics)} metric rows')

In [ ]:
# Plot 1: Semantic Similarity vs Score
sem_sims = [o['semantic_similarity_pct'] / 100 for o in outputs]
scores = [o['predicted_score'] for o in outputs]
types = [o['type'] for o in outputs]

type_colors = {
    'exact_match': 'green',
    'paraphrased': 'blue',
    'partial_answer': 'orange',
    'irrelevant_answer': 'red',
    'mixed_correctness': 'purple'
}
colors = [type_colors.get(t, 'gray') for t in types]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sem_sims, scores, c=colors, s=100, edgecolors='black', linewidths=0.5)
ax.set_xlabel('Semantic Similarity')
ax.set_ylabel('Predicted Score (out of 10)')
ax.set_title('Semantic Similarity vs Predicted Score')
ax.grid(True, alpha=0.3)

patches = [mpatches.Patch(color=v, label=k) for k, v in type_colors.items()]
ax.legend(handles=patches, loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('../results/graphs/similarity_vs_score.png', dpi=150)
plt.show()
print('Saved: results/graphs/similarity_vs_score.png')

In [ ]:
# Plot 2: Predicted vs Human Scores
predicted = [float(m['predicted_score']) for m in metrics]
human = [float(m['human_score']) for m in metrics]
ids = [m['id'] for m in metrics]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 10], [0, 10], 'k--', alpha=0.4, label='Perfect agreement')
ax.scatter(human, predicted, color='steelblue', s=100, edgecolors='black', linewidths=0.5, zorder=5)
for i, txt in enumerate(ids):
    ax.annotate(txt, (human[i], predicted[i]), textcoords='offset points', xytext=(5, 5), fontsize=7)
ax.set_xlabel('Human Score')
ax.set_ylabel('Predicted Score')
ax.set_title('Predicted vs Human Scores')
ax.set_xlim(-0.5, 11)
ax.set_ylim(-0.5, 11)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/graphs/predicted_vs_human.png', dpi=150)
plt.show()
print('Saved: results/graphs/predicted_vs_human.png')

In [ ]:
# Plot 3: Error Distribution
errors = [float(m['absolute_error']) for m in metrics]
mean_err = np.mean(errors)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(len(errors)), errors, color='salmon', edgecolor='black', linewidth=0.5)
ax.axhline(mean_err, color='red', linestyle='--', label=f'Mean MAE = {mean_err:.2f}')
ax.set_xticks(range(len(errors)))
ax.set_xticklabels([m['id'] for m in metrics], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Absolute Error')
ax.set_title('Error Distribution (Predicted vs Human)')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/graphs/error_distribution.png', dpi=150)
plt.show()
print('Saved: results/graphs/error_distribution.png')

In [ ]:
# Plot 4: Keyword Coverage vs Score
kw_covs = [o['keyword_coverage_pct'] / 100 for o in outputs]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(kw_covs, scores, c=colors, s=100, edgecolors='black', linewidths=0.5)
ax.set_xlabel('Keyword Coverage')
ax.set_ylabel('Predicted Score (out of 10)')
ax.set_title('Keyword Coverage vs Predicted Score')
ax.grid(True, alpha=0.3)
patches = [mpatches.Patch(color=v, label=k) for k, v in type_colors.items()]
ax.legend(handles=patches, loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('../results/graphs/keyword_vs_score.png', dpi=150)
plt.show()
print('Saved: results/graphs/keyword_vs_score.png')

In [ ]:
# Summary stats
print('=== Summary Statistics ===')
print(f'Total test cases: {len(outputs)}')
print(f'Mean predicted score: {np.mean(scores):.2f}')
print(f'Mean semantic similarity: {np.mean(sem_sims):.3f}')
print(f'Mean keyword coverage: {np.mean(kw_covs):.3f}')
print(f'Mean Absolute Error (vs human): {mean_err:.3f}')